In [1]:
from dotenv import load_dotenv
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
import os
import yaml
from pathlib import Path
import chromadb
import shutil
import hashlib
import numpy as np
import logging
from openai import OpenAIError

# Настройка логирования
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Пути и параметры
md_path = Path('c:/Users/Alkor/gd/news_rss_md_mix')
chromadb_path = './chroma_db_chatgpt_graph_mix'
model_name = "text-embedding-3-small"  # Модель OpenAI для эмбеддингов
batch_size = 5  # Размер батча для добавления документов

# Загрузка ключа API ChatGPT из .env файла
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

# Кастомная функция эмбеддингов для ChromaDB
class OpenAIEmbeddingFunction:
    def __init__(self, api_key, model_name):
        self.embeddings = OpenAIEmbeddings(api_key=api_key, model=model_name)
    
    def __call__(self, input):
        return self.embeddings.embed_documents(input)

def get_folder_size(folder_path):
    total_size = 0
    for dirpath, _, filenames in os.walk(folder_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024)  # Размер в МБ

def load_markdown_files(directory):
    documents = []
    for file_path in list(directory.glob("**/*.md")):
        # Чтение содержимого файла
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
        
        # Разделение метаданных и текста
        if content.startswith('---'):
            parts = content.split('---', 2)
            if len(parts) >= 3:
                metadata_yaml = parts[1].strip()
                text_content = parts[2].strip()
                # Парсинг метаданных
                metadata = yaml.safe_load(metadata_yaml)
                # Создание объекта Document
                doc = Document(
                    page_content=text_content,
                    metadata={
                        "next_bar": metadata.get("next_bar", ""), 
                        "source": file_path.name,
                        "date": file_path.stem
                    }
                )
                documents.append(doc)
            else:
                # Если нет метаданных, добавляем unknown
                doc = Document(
                    page_content=content,
                    metadata={
                        "next_bar": "unknown", 
                        "source": file_path.name,
                        "date": file_path.stem
                    }
                )
                documents.append(doc)
        else:
            # Если нет секции метаданных
            doc = Document(
                page_content=content,
                metadata={
                    "next_bar": "unknown", 
                    "source": file_path.name,
                    "date": file_path.stem
                }
            )
            documents.append(doc)
    return documents

# Функция для добавления документов батчами
def add_in_batches(collection, ids, documents, metadatas, batch_size):
    for i in range(0, len(documents), batch_size):
        batch_ids = ids[i:i + batch_size]
        batch_docs = documents[i:i + batch_size]
        batch_metas = metadatas[i:i + batch_size]
        try:
            logger.info(f"Добавление батча {i // batch_size + 1} с {len(batch_docs)} документами")
            collection.add(ids=batch_ids, documents=batch_docs, metadatas=batch_metas)
            logger.info(f"Батч {i // batch_size + 1} успешно добавлен")
        except OpenAIError as e:
            logger.error(f"Ошибка OpenAI API при добавлении батча {i // batch_size + 1}: {e}")
            raise
        except Exception as e:
            logger.error(f"Общая ошибка при добавлении батча {i // batch_size + 1}: {e}")
            raise

# Удаление папки chroma_db, если она существует
if os.path.exists(chromadb_path):
    print(f"Размер папки {chromadb_path} до удаления: {get_folder_size(chromadb_path):.2f} МБ")
    shutil.rmtree(chromadb_path)
    print(f"Папка {chromadb_path} удалена.")

# Инициализация клиента ChromaDB
client = chromadb.PersistentClient(path=chromadb_path)

# Создание функции эмбеддингов для OpenAI
ef = OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name=model_name
)

# Создание коллекции
collection = client.create_collection(name="news_collection", embedding_function=ef)

# Загрузка Markdown-файлов
documents = load_markdown_files(md_path)

# Проверка на пустую папку
if not documents:
    print("Не найдено Markdown-файлов в указанной директории.")
    exit(1)
else:
    print(f"Загружено {len(documents)} Markdown-файлов из {md_path}")
    print(f"Документы даты: {set(doc.metadata['date'] for doc in documents)}")
    print(f"Направление следующего бара: {set(doc.metadata['next_bar'] for doc in documents)}")

# Подготовка данных для ChromaDB
doc_texts = [doc.page_content for doc in documents]
doc_ids = [hashlib.md5(doc.page_content.encode()).hexdigest() for doc in documents]
doc_metadatas = [doc.metadata for doc in documents]

# Добавление в коллекцию батчами
try:
    add_in_batches(collection, doc_ids, doc_texts, doc_metadatas, batch_size)
    logger.info("Все документы успешно добавлены.")
except Exception as e:
    logger.error(f"Ошибка при добавлении документов: {e}")
    exit(1)

# # Пример поиска с фильтрацией по метаданным
# query = "Новости о Tesla"
# results = collection.query(
#     query_texts=[query],
#     n_results=3,
#     where={"next_bar": "up"}  # Фильтрация по next_bar
#     # where={"next_bar": "up", "date": {"$eq": "2025-07-24"}}
# )
# print(results)

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:__main__:Добавление батча 1 с 5 документами


Загружено 11 Markdown-файлов из c:\Users\Alkor\gd\news_rss_md_mix
Документы даты: {'2025-07-02', '2025-07-03', '2025-07-01', '2025-07-07', 'current', '2025-06-26', '2025-06-27', '2025-07-08', '2025-06-30', '2025-06-25', '2025-07-04'}
Направление следующего бара: {'up', 'current', 'down'}


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Батч 1 успешно добавлен
INFO:__main__:Добавление батча 2 с 5 документами
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Батч 2 успешно добавлен
INFO:__main__:Добавление батча 3 с 1 документами
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:__main__:Батч 3 успешно добавлен
INFO:__main__:Все документы успешно добавлены.


In [ ]:
from sklearn.manifold import TSNE
import numpy as np
import plotly.graph_objects as go

# Теперь мы можем визуализировать векторы с помощью t-SNE
# t-SNE - это метод, который позволяет визуализировать высокоразмерные данные
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['next_bar'] for metadata in metadatas]
colors = [['blue', 'red', 'black'][['up', 'down', 'current'].index(t)] for t in doc_types]


# Нам, людям, проще визуализировать объекты в 2D!
# Уменьшите размерность векторов до 2D, используя t-SNE
# (t-распределенное стохастическое вложение соседей)
tsne = TSNE(n_components=2, random_state=42, perplexity=3)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma OpenAI (text-embedding-3-small) 2025-07-10',
    xaxis_title='x',
    yaxis_title='y',
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [3]:
# Let's try 3D!
tsne = TSNE(n_components=3, random_state=42, perplexity=5)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()